In [14]:
from kokoro import KPipeline, KModel
import librosa
import soundfile as sf
import torch
from pathlib import Path
from typing import Optional
from pydantic import BaseModel, Field
from lingo_video_worker import *
from lingo_video_worker import _repo_dir, _resolve_voice_path

In [19]:
speech_text = "Hello, this is a test of Kokoro TTS in the LiveAvatar worker."
voice_id = "bf_isabella"
voice_id = 'af_sarah'
param = None

if param is None:
	param = KokoroSettings()

run_dir = scribble()

# Local imports keep worker startup light and only require Kokoro/Torch on TTS nodes.

output_path = (run_dir / "tts.wav").resolve()

repo_dir = _repo_dir()
voice_path = _resolve_voice_path(voice_id, repo_dir)
if not Path(voice_path).exists():
	raise FileNotFoundError(f"Kokoro voice file not found: {voice_path}")

model = None
# if param.model_path and param.model_path.exists():
# 	model = KModel().to('cuda' if torch.cuda.is_available() else 'cpu').eval()
# 	model.load_state_dict(torch.load(param.model_path, weights_only=True))
lang_code = param.language_code or voice_id[0]
print(f"Using language code '{lang_code}' for voice '{voice_id}'")
pipeline = KPipeline(lang_code=lang_code, repo_id=param.repo_id)
voice_tensor = torch.load(voice_path, weights_only=True)

chunks = []
generator = pipeline(
	speech_text,
	voice=voice_tensor,
	speed=param.speed,
	split_pattern=param.split_pattern,
)

for gs, ps, audio in generator:
	print(f"\n--- KOKORO DEBUG ---")
	print(f"Graphemes: {gs}")
	print(f"Phonemes: {ps}")
	print(f"Audio Output: {type(audio)}")
	if audio is not None:
		# Ensure it is a tensor before appending
		t = torch.tensor(audio) if not isinstance(audio, torch.Tensor) else audio
		chunks.append(t)

if not chunks:
	raise RuntimeError("Kokoro produced no audio samples")

merged = torch.cat(chunks, dim=0)

# Soundfile handles tensors natively if converted to numpy
sf.write(str(output_path), merged.numpy(), 24000)

# Normalize sample rate for downstream consistency.
wav_normalized, _ = librosa.load(str(output_path), sr=param.output_sample_rate)
sf.write(str(output_path), wav_normalized, param.output_sample_rate)

print(output_path)

Using language code 'a' for voice 'af_sarah'


/home/felix/miniconda3/envs/liveavatar/lib/python3.10/site-packages/torch/nn/modules/rnn.py:123: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn(
/home/felix/miniconda3/envs/liveavatar/lib/python3.10/site-packages/torch/nn/utils/weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)



--- KOKORO DEBUG ---
Graphemes: Hello, this is a test of Kokoro TTS in the LiveAvatar worker.
Phonemes: həlˈO, ðɪs ɪz ɐ tˈɛst ʌv kəkˈɔɹO tˌitˌiˈɛs ɪn ðə lˌIvˈævətˌɑɹ wˈɜɹkəɹ.
Audio Output: <class 'torch.Tensor'>
/tmp/scribble_f5u9ay4q/tts.wav
